# MLP — Rede neural do T2

Este notebook fica junto de `mlp.py`, como nos algoritmos do T1. Aqui vamos executar a rede por partes: definir a classe, criar os pesos, passar um tabuleiro e conferir as saídas.

A MLP escolhe onde jogar. O treinamento dos pesos fica no notebook do algoritmo genético. Não usamos backpropagation.

No VS Code, escolha o kernel da `.venv` e execute as células na ordem. As dependências estão em `requirements-notebook-t2.txt`, na raiz do projeto.

In [ ]:
from pathlib import Path
import sys
import json
from html import escape
import numpy as np
from IPython.display import HTML, display

# O kernel pode começar na pasta do notebook ou na raiz do projeto.
pasta_atual = Path.cwd().resolve()
raiz = next((p for p in [pasta_atual, *pasta_atual.parents] if (p / "app_t2.py").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Abra o notebook dentro do repositório.")
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from t2.treinamento import evaluate

In [ ]:
def mostrar_tabela(colunas, linhas):
    cabecalho = "".join(f"<th style='padding:8px;text-align:left'>{escape(str(c))}</th>" for c in colunas)
    corpo = "".join("<tr>" + "".join(f"<td style='padding:8px;border-top:1px solid #ddd'>{escape(str(v))}</td>" for v in linha) + "</tr>" for linha in linhas)
    display(HTML(f"<table><thead><tr>{cabecalho}</tr></thead><tbody>{corpo}</tbody></table>"))

## Implementação da rede

A classe abaixo tem o mesmo código de `mlp.py`. São 9 entradas, uma camada oculta com tanh e 9 saídas lineares. Os bias também ficam no cromossomo.

In [ ]:
class Network:
    """
    A rede recebe 9 entradas e passa por H neurônios ocultos com tanh.

    No final temos 9 saídas lineares, uma para cada casa. Tanto a camada
    oculta quanto a de saída têm bias, que também entram no cromossomo.
    """
    def __init__(self, weights, hidden=9):
        self.hidden = hidden
        self.weights = np.asarray(weights, dtype=float)
        if self.weights.shape != (self.size(hidden),) or not np.isfinite(self.weights).all():
            raise ValueError('Cromossomo incompatível com a topologia.')
        cut = 9 * hidden
        self.w1 = self.weights[:cut].reshape(9, hidden)
        self.b1 = self.weights[cut:cut+hidden]
        cut += hidden
        self.w2 = self.weights[cut:cut+hidden*9].reshape(hidden, 9)
        self.b2 = self.weights[-9:]

    @staticmethod
    def size(hidden):
        return 9*hidden + hidden + hidden*9 + 9

    def forward(self, board, player):
        # A entrada precisa ter o mesmo sentido quando a rede joga como X
        # ou como O. Por isso a própria peça vale 1, a do adversário vale -1
        # e a casa vazia continua valendo 0.
        inputs = np.asarray(board, dtype=float) * player
        activation = np.tanh(inputs @ self.w1 + self.b1)
        return activation @ self.w2 + self.b2

    def choose(self, board, player):
        outputs = self.forward(board, player)
        ranking = np.argsort(-outputs, kind='stable')
        raw = int(ranking[0])
        # Guardamos a primeira escolha mesmo se a casa estiver ocupada,
        # porque é esse erro que vai receber a penalidade na avaliação.
        # Para a partida não travar ali, procuramos a próxima casa livre
        # na ordem das saídas da própria rede.
        move = next(int(i) for i in ranking if board[i] == 0)
        return raw, move, outputs.tolist()

## Pesos iniciais

Vamos criar uma rede com pesos sorteados entre -1 e 1. Com a camada oculta de 9 neurônios, são 180 parâmetros.

In [ ]:
ocultos = 9
semente = 42
rng = np.random.default_rng(semente)
pesos = rng.uniform(-1, 1, Network.size(ocultos))
rede_inicial = Network(pesos, ocultos)
print("Parâmetros:", len(pesos))
mostrar_tabela(["Camada", "Formato"], [
    ["W1", rede_inicial.w1.shape], ["b1", rede_inicial.b1.shape],
    ["W2", rede_inicial.w2.shape], ["b2", rede_inicial.b2.shape],
])

## Propagação de um tabuleiro

A própria peça vale 1, a do adversário vale -1 e uma casa vazia vale 0. O parâmetro `jogador` indica de quem é a vez. A maior saída é a escolha original da rede, que ainda pode ser uma casa ocupada.

In [ ]:
tabuleiro = [1, -1, 0, 0, 1, 0, -1, 0, 0]
jogador = 1
original, executada, saidas = rede_inicial.choose(tabuleiro, jogador)
mostrar_tabela(["Casa", "Ocupação", "Saída"], [[i+1, tabuleiro[i], saidas[i]] for i in range(9)])
print("Escolha original:", original+1)
print("Casa executada:", executada+1)
print("Escolha original ocupada:", tabuleiro[original] != 0)

## Rede com os pesos do experimento

Agora carregamos os pesos que o AG encontrou. A estrutura da MLP é a mesma, o que muda são os valores do cromossomo. Não estamos treinando novamente nesta célula.

In [ ]:
salva = json.loads((raiz / "t2" / "resultados" / "rede-t2.json").read_text())
rede_treinada = Network(salva["weights"], salva["hidden"])
original, executada, saidas = rede_treinada.choose(tabuleiro, jogador)
mostrar_tabela(["Casa", "Saída da rede treinada"], [[i+1, valor] for i, valor in enumerate(saidas)])
print("Escolha original:", original+1)
print("Casa executada:", executada+1)

## Avaliação das escolhas

Usamos o teste reservado para olhar as decisões da rede carregada. A acurácia verifica se a escolha original está entre as ações ótimas do Minimax. A recuperação de uma posição inválida não transforma essa escolha em acerto.

In [ ]:
resultados = {
    "Médio": evaluate(rede_treinada, 100, False, 1000000, True),
    "Difícil": evaluate(rede_treinada, 0, True, 2000000, True),
}
mostrar_tabela(["Modo", "Vitórias", "Empates", "Derrotas", "Inválidas", "Acurácia"], [
    [modo, r["vitoria"], r["empate"], r["derrota"], r["invalid"], f'{100*r["accuracy"]:.1f}%']
    for modo, r in resultados.items()
])

## Relação com os outros algoritmos

A rede só calcula a jogada com os pesos que recebeu. Para acompanhar como esses pesos são escolhidos, abra [genetico.ipynb](../genetico/genetico.ipynb). O adversário usado na avaliação está em [minimax.ipynb](../minimax/minimax.ipynb).

Os arquivos `.py` continuam sendo usados pelo Flask. Se alterar a implementação no notebook, essa alteração não muda automaticamente o arquivo `.py`.